
# Supplementary 01 — Training history forensics

Deep audit of the 82 recovered `history.csv` logs: completeness, epoch
contiguity, selection provenance and the metrics each run actually recorded.

Companion to **§04–§05** of `ReMedFiLM_Complete_Research_Master.ipynb`.


In [1]:

import sys
from pathlib import Path
sys.path.insert(0, r"D:\Softfors\jup\full_doc_report\Scripts")
import remedfilm_guard as GUARD
GUARD.install()

REPO = Path(r"D:\Softfors\jup")
REPORT = REPO / "full_doc_report"
TABLES = REPORT / "tables"
FIGURES = REPORT / "figures"

import pandas as pd
import numpy as np
import json
from IPython.display import Image, display

for name, verdict in GUARD.self_test():
    assert verdict.startswith("blocked"), f"GUARD FAILED: {name} {verdict}"
print("analysis-only guard verified")


analysis-only guard verified


In [2]:

reg = pd.read_csv(TABLES / "TRAINING_RUN_REGISTRY.csv")
prov = json.loads((REPORT / "TRAINING_HISTORY_PROVENANCE.json").read_text(encoding="utf-8"))
print("history files :", prov["n_history_files"])
print("runs          :", prov["n_runs"])
print("epoch records :", prov["n_epoch_records"])
print()
print("epoch contiguity (no gaps, no duplicates):")
non_contig = [r for r in prov["runs"] if not r["epoch_contiguous"]]
print(f"  contiguous runs     : {len(prov['runs']) - len(non_contig)}")
print(f"  non-contiguous runs : {len(non_contig)}")
for r in non_contig[:6]:
    print("   ", r["run_id"], r["epoch_min"], "->", r["epoch_max"], f"({r['n_epoch_records']} rows)")


history files : 82
runs          : 82
epoch records : 35764

epoch contiguity (no gaps, no duplicates):
  contiguous runs     : 82
  non-contiguous runs : 0


In [3]:

# Duplicate epoch detection inside each log.
hist = pd.read_csv(TABLES / "REMEDFILM_EPOCH_HISTORY.csv")
dups = (hist.groupby(["run_id", "metric", "epoch"]).size()
            .reset_index(name="n").query("n > 1"))
print("duplicated (run, metric, epoch) rows:", len(dups))
if len(dups):
    print(dups.head(10).to_string(index=False))


duplicated (run, metric, epoch) rows: 0


In [4]:

# Selection provenance: recorded vs derived.
print(reg.best_epoch_source.value_counts().to_string())
print()
derived = reg[reg.best_epoch_source.str.startswith("DERIVED")]
print(f"runs whose best epoch is DERIVED by analysis: {len(derived)}")
derived[["run_id", "model", "best_epoch", "selection_metric"]].head(12)


best_epoch_source
RECORDED_FLAG                       37
RECORDED_CLOSURE                    19
NOT_RECORDED                        14
DERIVED_ARGMAX                       7
NOT_APPLICABLE_FIXED_EPOCH_REFIT     4
RECORDED_EVENT                       1

runs whose best epoch is DERIVED by analysis: 7


,run_id,model,best_epoch,selection_metric
1,phase10_kub_attention_unet_seed42_512,KUB baseline,77,argmax val_dice
2,phase10_kub_attention_unet_seed42_512_clean_v3,KUB baseline,54,argmax val_dice
3,phase10_kub_deeplabv3plus_seed42_512,KUB baseline,37,argmax val_dice
4,phase10_kub_deeplabv3plus_seed42_512_clean_v3,KUB baseline,52,argmax val_dice
5,phase10_kub_unetpp_seed42_512,KUB baseline,77,argmax val_dice
6,phase10_kub_unetpp_seed42_512_clean_v3,KUB baseline,70,argmax val_dice
77,phase9_kub_unet_baseline_seed42_512,U-Net,47,argmax val_dice


In [5]:

# Metric completeness matrix: which canonical metric each run logged.
canon = ["train_loss", "val_loss", "val_macro_dice", "val_dice", "val_iou",
         "val_precision", "val_recall", "learning_rate"]
present = (hist.assign(one=1)
               .pivot_table(index="run_id", columns="metric", values="one",
                            aggfunc="max", fill_value=0))
cols = [c for c in canon if c in present.columns]
mat = present[cols]
print("runs:", len(mat))
print("share of runs logging each metric:")
print((mat.mean() * 100).round(1).to_string())
mat.head(10)


runs: 82
share of runs logging each metric:
metric
train_loss         56.1
val_loss           32.9
val_macro_dice      8.5
val_dice            8.5
val_iou             8.5
val_precision       8.5
val_recall          8.5
learning_rate     100.0


metric,train_loss,val_loss,val_macro_dice,val_dice,val_iou,val_precision,val_recall,learning_rate
run_id,,,,,,,,
h3_kub_localizer_sigma_adaptive_sizeaware_seed42_clean_v3,1,1,0,0,0,0,0,1
phase10_kub_attention_unet_seed42_512,1,1,0,1,1,1,1,1
phase10_kub_attention_unet_seed42_512_clean_v3,1,1,0,1,1,1,1,1
phase10_kub_deeplabv3plus_seed42_512,1,1,0,1,1,1,1,1
phase10_kub_deeplabv3plus_seed42_512_clean_v3,1,1,0,1,1,1,1,1
phase10_kub_unetpp_seed42_512,1,1,0,1,1,1,1,1
phase10_kub_unetpp_seed42_512_clean_v3,1,1,0,1,1,1,1,1
phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_clean_v3,1,1,0,0,0,0,0,1
phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_v1,1,1,0,0,0,0,0,1


In [6]:

# Per-domain coverage — only the four-domain runs log per-domain metrics.
dom_cols = [c for c in present.columns if c.startswith("val_dice_")]
print("per-domain metrics available:", dom_cols)
runs_with_dom = present.index[present[dom_cols].sum(axis=1) > 0].tolist() if dom_cols else []
print(f"runs logging per-domain Dice: {len(runs_with_dom)}")
for r in runs_with_dom:
    print("  ", r)


per-domain metrics available: ['val_dice_BUSI', 'val_dice_ISIC', 'val_dice_KUB', 'val_dice_KVASIR']
runs logging per-domain Dice: 6
   phase20_shared_unetpp_fourdomain_seed42_clean_v3
   phase20b_shared_attention_unet_fourdomain_seed42_clean_v3
   phase20b_shared_deeplabv3plus_fourdomain_seed42_clean_v3
   phase20b_shared_transformer_mit_unet_fourdomain_seed42_clean_v3
   phase20b_shared_unet_fourdomain_seed42_clean_v3
   phase21_global_medfilm_fourdomain_seed42_clean_v3


In [7]:

# The single most complete log.
g = hist[hist.run_id == "phase21_global_medfilm_fourdomain_seed42_clean_v3"]
piv = g.pivot_table(index="epoch", columns="metric", values="value")
print("GlobalMedFiLM: epochs =", len(piv), "| metrics =", piv.shape[1])
print("columns:", list(piv.columns))
print()
print("last 5 epochs:")
piv.tail(5)[["train_loss", "val_loss", "val_macro_dice", "learning_rate"]]


GlobalMedFiLM: epochs = 81 | metrics = 23
columns: ['best_epoch_so_far', 'best_metric_so_far', 'epoch_seconds', 'learning_rate', 'train_loss', 'val_dice_BUSI', 'val_dice_ISIC', 'val_dice_KUB', 'val_dice_KVASIR', 'val_iou_BUSI', 'val_iou_ISIC', 'val_iou_KUB', 'val_iou_KVASIR', 'val_loss', 'val_macro_dice', 'val_precision_BUSI', 'val_precision_ISIC', 'val_precision_KUB', 'val_precision_KVASIR', 'val_recall_BUSI', 'val_recall_ISIC', 'val_recall_KUB', 'val_recall_KVASIR']

last 5 epochs:


metric,train_loss,val_loss,val_macro_dice,learning_rate
epoch,,,,
77,0.057570,0.150547,0.832780,0.000012
78,0.056881,0.149761,0.832451,0.000011
79,0.057313,0.152856,0.831825,0.000010
80,0.055589,0.149932,0.832839,0.000010
81,0.056010,0.150661,0.831697,0.000009


In [8]:

# Negative control: enforcement must be real.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test())
print("OK")


  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
OK
